# 00 — Push the system prompt to all 3 registries

Each of the three platforms has a **prompt management** feature — a versioned registry for prompts your application uses. This notebook pushes our agent's `SYSTEM_PROMPT` (defined in `shared/workflow.py`) into all three so that notebooks 01 / 02 / 03 can **pull the prompt at runtime** instead of importing the Python constant.

Run this **once when setting up from scratch** (or any time the prompt changes). It's idempotent — re-running is safe.

Each section is intentionally a separate cell so you can:
- See each platform's API surface side-by-side
- Skip a platform if its account isn't ready yet
- Use this notebook itself as one of the comparison slides (it shows the API differences concretely)

## What you'll see at the end

| Platform | UI location after push |
| --- | --- |
| 🔵 Langfuse | `LANGFUSE_HOST` → your project → **Prompts** tab |
| 🟢 LangSmith | <https://smith.langchain.com> → **Prompts** (top nav, workspace-level — NOT inside a project) |
| 🟠 Galileo | <https://app.galileo.ai> → project from `GALILEO_PROJECT` → **Templates** / **Prompts** |

All project and host names are read from your `.env`. See `.env.example` for the defaults.

## 1. Load environment + the SYSTEM_PROMPT we want to register

In [ ]:
import os, sys, pathlib
from dotenv import load_dotenv

ROOT = pathlib.Path().resolve().parent
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

load_dotenv(ROOT / ".env")

from shared.workflow import SYSTEM_PROMPT, PROMPT_NAME

print(f"PROMPT_NAME : {PROMPT_NAME}")
print(f"SYSTEM_PROMPT: {len(SYSTEM_PROMPT)} chars")
print("---")
print(SYSTEM_PROMPT[:300] + "...")

## 2. 🔵 Langfuse — `lf.create_prompt(...)`

Langfuse models a prompt as a **named entity with sequential versions** (v1, v2, …). Each version can carry **labels** like `production` or `staging` — labels are mutable pointers, so promoting a version is a label move, not a redeploy.

**Idempotency**: Langfuse's `create_prompt` always creates a new version, so naive re-runs would accumulate identical v1, v2, v3, … This cell first **checks the current `production` version** and only pushes if the content differs from `SYSTEM_PROMPT` — making the notebook safe to re-run.

In [ ]:
from langfuse import Langfuse

lf = Langfuse()
assert lf.auth_check(), "Langfuse auth failed — check LANGFUSE_PUBLIC_KEY / SECRET_KEY / HOST"

# Content-aware idempotency: only push if the production version is missing
# or its content differs from SYSTEM_PROMPT.
needs_push = True
try:
    current = lf.get_prompt(PROMPT_NAME, label="production")
    if current.prompt == SYSTEM_PROMPT:
        needs_push = False
        print(f"  Langfuse: `{PROMPT_NAME}` v{current.version} already matches — skipping push")
    else:
        print(f"  Langfuse: production is v{current.version}, content differs — pushing new version")
except Exception:
    # Not found at all
    print(f"  Langfuse: `{PROMPT_NAME}` not found — pushing v1")

if needs_push:
    created = lf.create_prompt(
        name=PROMPT_NAME,
        type="text",                                       # could be "chat" for multi-message templates
        prompt=SYSTEM_PROMPT,
        labels=["production"],                              # serves as the default for `lf.get_prompt(name, label="production")`
        tags=["observability_comparison", "permission-agent"],
        commit_message="Pushed by notebook 00_setup_prompts",
    )
    print(f"  Langfuse: pushed `{PROMPT_NAME}` v{created.version}")

print(f"  → Open {os.environ.get('LANGFUSE_HOST', 'http://localhost:3000')} → Prompts tab")

## 3. 🟢 LangSmith — `client.push_prompt(...)`

LangSmith stores prompts as **LangChain Runnable objects** (typically `ChatPromptTemplate`), not raw strings. Versioning is **git-style**: each push creates a commit with a hash, and pushing identical content raises **409 Conflict** ("Nothing to commit"). We catch and report that case as success.

This is the tightest integration with LangChain itself — the prompt object you pull back is directly pipeable into your chain.

In [ ]:
from langsmith import Client
from langchain_core.prompts import ChatPromptTemplate

client = Client()

chat_prompt = ChatPromptTemplate.from_messages([
    ("system", SYSTEM_PROMPT),
    ("placeholder", "{messages}"),
])

try:
    url = client.push_prompt(
        PROMPT_NAME,
        object=chat_prompt,
        description="Permission-checking helpdesk agent — system prompt",
        tags=["observability_comparison", "permission-agent"],
        is_public=False,
    )
    print(f"  LangSmith: pushed `{PROMPT_NAME}`")
    print(f"  URL: {url}")
except Exception as e:
    if "Nothing to commit" in str(e):
        print(f"  LangSmith: `{PROMPT_NAME}` already up to date (no changes since latest commit)")
    else:
        raise

## 4. 🟠 Galileo — `create_prompt(...)` from `galileo.prompts`

Galileo models a prompt as a **list of typed `Message` objects** — no raw-string format, chat shape only. Templates are **project-scoped** (unlike Langfuse / LangSmith, which are workspace-global), so the template lives inside whatever project name `GALILEO_PROJECT` is set to in your `.env`.

**Idempotency**: like Langfuse, naive re-runs would accumulate templates. This cell uses `get_prompt` first to detect a matching existing template and only calls `create_prompt` if missing or content differs.

In [ ]:
import json as _json
from galileo.prompts import create_prompt, get_prompt, Message
from galileo_core.schemas.logging.llm import MessageRole

project = os.environ.get("GALILEO_PROJECT", "observability-comparison")

def _galileo_extract_system(template) -> str:
    """Extract the system-role content from a Galileo template (which the
    API returns as a JSON string of message dicts, not typed objects)."""
    version = getattr(template, "selected_version", None) or getattr(template, "version", None)
    raw = getattr(version, "template", None) if version else None
    if isinstance(raw, str):
        try:
            messages = _json.loads(raw)
        except Exception:
            return ""
    elif isinstance(raw, list):
        messages = raw
    else:
        return ""
    for m in messages:
        role = m.get("role") if isinstance(m, dict) else getattr(m, "role", "")
        if str(role).lower().endswith("system"):
            content = m.get("content") if isinstance(m, dict) else getattr(m, "content", None)
            if isinstance(content, str):
                return content
    return ""

# Content-aware: skip push if a template with this name already holds the same content.
needs_push = True
existing = get_prompt(name=PROMPT_NAME, project_name=project)
if existing and _galileo_extract_system(existing) == SYSTEM_PROMPT:
    needs_push = False
    print(f"  Galileo: `{PROMPT_NAME}` already matches in project `{project}` — skipping push")

if needs_push:
    template = create_prompt(
        name=PROMPT_NAME,
        template=[Message(role=MessageRole.system, content=SYSTEM_PROMPT)],
        project_name=project,
    )
    print(f"  Galileo: pushed `{PROMPT_NAME}` to project `{project}`")
    print(f"  template id: {template.id}")

print(f"  → Open https://app.galileo.ai → {project} → Templates / Prompts")

## 5. Verify all three by pulling back via `fetch_system_prompt`

Sanity check: each platform should return the same 954-char prompt we just pushed. If any of them drift in length, something went wrong.

In [ ]:
from shared.workflow import fetch_system_prompt

for src in ["local", "langfuse", "langsmith", "galileo"]:
    p = fetch_system_prompt(src)
    head = p.splitlines()[0][:60]
    print(f"  {src:<10} len={len(p):>4}  starts={head!r}")

## What's different between the three APIs (cheat-sheet)

| | Langfuse | LangSmith | Galileo |
| --- | --- | --- | --- |
| **SDK call** | `lf.create_prompt(name, prompt=, ...)` | `client.push_prompt(name, object=runnable)` | `create_prompt(name, template=[Message(...)])` |
| **Format** | Raw string OR chat messages | LangChain Runnable | Chat messages only |
| **Versioning** | Sequential `v1`, `v2`, … | Git-style commit hashes | Sequential versions |
| **Scoping** | Project-global | Workspace-global | **Project-scoped** |
| **Idempotency** | Always creates new version | 409 if unchanged | Always creates new version |
| **Default-version pointer** | Mutable **label** (e.g. `production`) | Tag a hash | Latest (no label system) |

## Next step

Open notebooks 01 / 02 / 03 — they each call `build_agent(prompt_source="<platform>")` which pulls the prompt back from the registry. Run them and the traces will land in each platform with the prompt-managed-by-platform story intact.